### TOOLS

Models can requres to call tools that perform tasks such as fetching data from a database, searching the web, or running code. 
Tools are pairing of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or co-routine to execute

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() 

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why parrort talks?")

response 


'Parrots aren’t “talking” in the way humans do—they’re amazing vocal mimics.  \nHere’s why many of them can reproduce words and short phrases:\n\n| Factor | What it is | How it helps a parrot “talk” |\n|--------|------------|------------------------------|\n| **Vocal anatomy – the syrinx** | Birds have a voice box called a *syrinx* at the base of the trachea. In parrots the syrinx is especially well‑developed and can produce a very wide range of tones and rapid modulations. | This gives them the fine motor control needed to copy the pitch, rhythm, and timbre of human speech. |\n| **Brain structures for vocal learning** | Parrots belong to the *psittaciformes* order, which has a highly specialized set of brain nuclei (the “song system”) that are analogous to those used by songbirds and humans for learning complex vocalizations. | These nuclei let them form a mental map of sounds they hear, store them, and later reproduce them. |\n| **Social nature** | In the wild parrots live in tight‑k

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str :
    """Get the weather at a location"""
    print(f"It's sunny in {location}")

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What's the weather like in india?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")


content='' additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather like in india?" We need to get weather. The function get_weather expects a location string. Could use "India". Use function.', 'tool_calls': [{'id': 'fc_097bed4b-26c5-46bb-bf5f-3d95d9def10a', 'function': {'arguments': '{"location":"India"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 127, 'total_tokens': 189, 'completion_time': 0.12900608, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.005265562, 'prompt_tokens_details': None, 'queue_time': 0.212655806, 'total_time': 0.134271642}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0e402-f45d-7ab0-875e-3522e70eb82d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'India'}, 'id': 'fc_097bed4b-26c5

### Tool Exectuion Loops

In [ ]:
messages = [{"role": "user", "content": "What's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)